In [ ]:
# ============================================================
# BranchyNet + Adaptive Computation — ViT (patch4/32) on CIFAR-10
# Early-Exit Inference with Confidence-Based Adaptive Depth
# Compatible with baseline: __4__baseline_vit_pretrained_cifar10.pth
# Output: __5__branchynet_vit_metrics.json
# ============================================================

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import timm
import time, os, json, tempfile
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from thop import profile as thop_profile

In [ ]:
# ── REPRODUCIBILITY ───────────────────────────────────────────
SEED = 42
import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
EPOCHS      = 30
NUM_CLASSES = 10
IMG_SIZE    = 32

BASELINE_WEIGHTS = "../__4__baseline_vit_pretrained_cifar10.pth"
SAVE_PATH        = "__5__branchynet_vit_cifar10.pth"

# Exit confidence thresholds to sweep during evaluation
EXIT_THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]

# Branch loss weights: [exit1 (shallow), exit2 (mid), final (deep)]
# Deeper head dominates — mirrors ResNet BranchyNet strategy
BRANCH_WEIGHTS = [0.2, 0.3, 0.5]

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"Using device: {DEVICE}")

In [4]:
# ── AUXILIARY BRANCH (early-exit classifier head) ─────────────
class EarlyExitBranch(nn.Module):
    """
    Tiny classifier attached to an intermediate ViT hidden state.
    input_dim  : hidden dimension of the tapped transformer layer
    num_classes: number of output classes
    """
    def __init__(self, input_dim: int, num_classes: int = 10):
        super().__init__()
        self.branch = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        # x: (B, N, D) — take CLS token (index 0) for classification
        cls = x[:, 0]        # (B, D)
        return self.branch(cls)

In [5]:
# ── BRANCHYNET ViT ────────────────────────────────────────────
class BranchyViT(nn.Module):
    """
    ViT-Small (patch4, img32) with two auxiliary early-exit branches
    inserted after block[3] (shallow) and block[7] (mid-depth).
    The final head of the original ViT is the third (deepest) exit.

    Architecture:
      12 transformer blocks total (vit_small_patch16_224)
      Exit 1 → after block  3   (~33% of depth)
      Exit 2 → after block  7   (~67% of depth)
      Exit 3 → after block 11   (full depth, original head)
    """
    def __init__(self, num_classes: int = 10):
        super().__init__()

        # Build the backbone configured for 32×32 / patch_size=4
        backbone = timm.create_model(
            'vit_small_patch16_224',
            pretrained=False,
            num_classes=num_classes,
            img_size=IMG_SIZE,
            patch_size=4,
        )

        # ── Expose sub-modules we need ─────────────────────────
        self.patch_embed = backbone.patch_embed
        self.cls_token   = backbone.cls_token
        self.pos_embed   = backbone.pos_embed
        self.pos_drop    = backbone.pos_drop
        self.blocks      = backbone.blocks      # nn.Sequential of 12 blocks
        self.norm        = backbone.norm
        self.head        = backbone.head

        hidden_dim = backbone.embed_dim          # 384 for vit_small

        # ── Two auxiliary exit branches ────────────────────────
        self.branch1 = EarlyExitBranch(hidden_dim, num_classes)   # after block 3
        self.branch2 = EarlyExitBranch(hidden_dim, num_classes)   # after block 7

        self.exit1_block_idx = 3
        self.exit2_block_idx = 7

    # ── Load pretrained backbone weights ──────────────────────
    def load_backbone(self, path: str, device):
        state = torch.load(path, map_location=device)
        # Map flat state-dict keys from the timm model to our re-exposed submodules
        own_state = self.state_dict()
        loaded, skipped = 0, 0
        for k, v in state.items():
            if k in own_state and own_state[k].shape == v.shape:
                own_state[k] = v
                loaded += 1
            else:
                skipped += 1
        self.load_state_dict(own_state)
        print(f"  Backbone weights loaded: {loaded} tensors  (skipped/new: {skipped})")

    # ── Full forward — all three exits (used during training) ──
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)                                # (B, 64, D)
        cls = self.cls_token.expand(B, -1, -1)                 # (B,  1, D)
        x   = torch.cat([cls, x], dim=1)                       # (B, 65, D)
        x   = self.pos_drop(x + self.pos_embed)

        for i, blk in enumerate(self.blocks):
            x = blk(x)
            if i == self.exit1_block_idx:
                logits1 = self.branch1(x)
            elif i == self.exit2_block_idx:
                logits2 = self.branch2(x)

        x       = self.norm(x)
        logits3 = self.head(x[:, 0])

        return logits1, logits2, logits3

    # ── Adaptive forward — true early stopping ─────────────────
    # Stops as soon as the entire batch is confident enough.
    # Per-sample exit assignments are still recorded for analysis.
    @torch.no_grad()
    def adaptive_forward(self, x, threshold: float = 0.8):
        """
        True early-stopping forward pass.

        Halts at the first exit whose max-softmax confidence meets
        `threshold` for ALL samples in the batch.  Deeper blocks are
        never executed when the whole batch exits early.

        Returns
        -------
        logits   : Tensor (B, num_classes)
        exit_idx : Tensor (B,) — 0 = branch1, 1 = branch2, 2 = final
        """
        B = x.shape[0]
        h = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        h   = torch.cat([cls, h], dim=1)
        h   = self.pos_drop(h + self.pos_embed)

        # ── Blocks 0–exit1 ───────────────────────────────────
        for i in range(self.exit1_block_idx + 1):
            h = self.blocks[i](h)

        logits1 = self.branch1(h)
        conf1   = torch.softmax(logits1, dim=1).max(dim=1).values

        if (conf1 >= threshold).all():
            exit_idx = torch.zeros(B, dtype=torch.long, device=x.device)
            return logits1, exit_idx

        # ── Blocks exit1+1–exit2 ─────────────────────────────
        for i in range(self.exit1_block_idx + 1, self.exit2_block_idx + 1):
            h = self.blocks[i](h)

        logits2 = self.branch2(h)
        conf2   = torch.softmax(logits2, dim=1).max(dim=1).values

        if (conf2 >= threshold).all():
            exit_idx = torch.ones(B, dtype=torch.long, device=x.device)
            return logits2, exit_idx

        # ── Remaining blocks → final head ────────────────────
        for i in range(self.exit2_block_idx + 1, len(self.blocks)):
            h = self.blocks[i](h)

        h       = self.norm(h)
        logits3 = self.head(h[:, 0])

        # Per-sample exit bookkeeping (for analysis, even when we reach here)
        exit_idx = torch.full((B,), 2, dtype=torch.long, device=x.device)
        exit_idx[conf2 >= threshold] = 1
        exit_idx[conf1 >= threshold] = 0

        # Use best available logits per sample
        final_logits = logits3.clone()
        mask2 = conf2 >= threshold
        final_logits[mask2] = logits2[mask2]
        mask1 = conf1 >= threshold
        final_logits[mask1] = logits1[mask1]

        return final_logits, exit_idx

In [ ]:
# ── BUILD & LOAD BACKBONE ─────────────────────────────────────
model = BranchyViT(num_classes=NUM_CLASSES).to(DEVICE)
model.load_backbone(BASELINE_WEIGHTS, DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters (BranchyViT): {total_params:,}")

In [ ]:
# ── DATA ─────────────────────────────────────────────────────
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    transforms.RandomErasing(p=0.25),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_set = torchvision.datasets.CIFAR10(root='../data', train=True,
                                          download=True, transform=transform_train)
test_set  = torchvision.datasets.CIFAR10(root='../data', train=False,
                                          download=True, transform=transform_test)

train_loader = torch.utils.data.DataLoader(
    train_set, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, worker_init_fn=seed_worker, generator=g, pin_memory=True,
)
test_loader = torch.utils.data.DataLoader(
    test_set, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True,
)

print(f"Train: {len(train_set)} | Test: {len(test_set)}")

In [ ]:
# ── TRAINING ──────────────────────────────────────────────────
WARMUP_EPOCHS = 3
LR            = 3e-4

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def branchynet_loss(logits_list, labels, weights=BRANCH_WEIGHTS):
    assert len(logits_list) == len(weights)
    return sum(w * criterion(l, labels) for w, l in zip(weights, logits_list))

# Higher LR for branch heads (trained from scratch), lower for backbone
branch_params     = list(model.branch1.parameters()) + list(model.branch2.parameters())
branch_param_ids  = {id(p) for p in branch_params}
backbone_params   = [p for p in model.parameters() if id(p) not in branch_param_ids]

optimizer = torch.optim.AdamW([
    {'params': branch_params,   'lr': LR * 5},    # branches: higher LR
    {'params': backbone_params, 'lr': LR},          # backbone: standard LR
], weight_decay=0.05)

def get_lr(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / (EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr)


def train_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits1, logits2, logits3 = model(inputs)
        loss = branchynet_loss([logits1, logits2, logits3], labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        correct    += logits3.argmax(1).eq(labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total


def evaluate_standard(model, loader):
    """Accuracy using only the final exit head."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            _, _, logits3 = model(inputs)
            correct += logits3.argmax(1).eq(labels).sum().item()
            total   += labels.size(0)
    return correct / total


best_val_acc = 0.0
train_accs, val_accs, train_losses = [], [], []

print("\n" + "="*60)
print("TRAINING — BranchyViT (3 exits)")
print("="*60)

for epoch in range(EPOCHS):
    loss, train_acc = train_epoch(model, train_loader, optimizer)
    val_acc         = evaluate_standard(model, test_loader)
    scheduler.step()

    current_lr = optimizer.param_groups[1]['lr']
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    train_losses.append(loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        marker = " ← best saved"
    else:
        marker = ""

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | LR: {current_lr:.6f} | Loss: {loss:.4f} | "
          f"Train: {train_acc:.4f} | Val: {val_acc:.4f}{marker}")

print(f"\nBest validation accuracy (final exit): {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")

In [ ]:
# ── FULL EVALUATION (standard — final exit only) ──────────────
print("\n" + "="*60)
print("FULL EVALUATION — final exit (no early stopping)")
print("="*60)

model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        _, _, logits3 = model(inputs)
        all_preds.extend(logits3.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc_final  = accuracy_score(all_labels, all_preds)
prec_final = precision_score(all_labels, all_preds, average='macro', zero_division=0)
rec_final  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
f1_final   = f1_score(all_labels, all_preds, average='macro', zero_division=0)

print(f"  Accuracy          : {acc_final:.4f}")
print(f"  Precision (macro) : {prec_final:.4f}")
print(f"  Recall    (macro) : {rec_final:.4f}")
print(f"  F1-score  (macro) : {f1_final:.4f}")

In [ ]:
# ── ADAPTIVE COMPUTATION EVALUATION ──────────────────────────
print("\n" + "="*60)
print("ADAPTIVE COMPUTATION — Early-Exit Threshold Sweep")
print("="*60)
print(f"  Thresholds tested: {EXIT_THRESHOLDS}")

adaptive_results = []

for tau in EXIT_THRESHOLDS:
    preds_list, labels_list, exit_idx_list = [], [], []

    t_start = time.time()
    model.eval()
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(DEVICE)
            logits, exit_idx = model.adaptive_forward(inputs, threshold=tau)
            preds_list.extend(logits.argmax(1).cpu().numpy())
            labels_list.extend(labels.numpy())
            exit_idx_list.extend(exit_idx.cpu().numpy())
    t_end = time.time()

    preds_arr    = np.array(preds_list)
    labels_arr   = np.array(labels_list)
    exit_idx_arr = np.array(exit_idx_list)
    n            = len(labels_arr)

    acc   = accuracy_score(labels_arr, preds_arr)
    prec  = precision_score(labels_arr, preds_arr, average='macro', zero_division=0)
    rec   = recall_score(labels_arr, preds_arr, average='macro', zero_division=0)
    f1    = f1_score(labels_arr, preds_arr, average='macro', zero_division=0)

    frac_exit1  = (exit_idx_arr == 0).mean()
    frac_exit2  = (exit_idx_arr == 1).mean()
    frac_exit3  = (exit_idx_arr == 2).mean()
    avg_time_ms = (t_end - t_start) / n * 1000

    adaptive_results.append({
        "threshold"  : tau,
        "accuracy"   : round(acc,  6),
        "precision"  : round(prec, 6),
        "recall"     : round(rec,  6),
        "f1"         : round(f1,   6),
        "frac_exit1" : round(frac_exit1, 4),
        "frac_exit2" : round(frac_exit2, 4),
        "frac_exit3" : round(frac_exit3, 4),
        "avg_time_ms": round(avg_time_ms, 4),
    })

    print(f"  τ={tau:.2f} | Acc={acc:.4f} | F1={f1:.4f} | "
          f"Exit1={frac_exit1:.1%} Exit2={frac_exit2:.1%} Exit3={frac_exit3:.1%} | "
          f"Time={avg_time_ms:.4f}ms/sample")

In [ ]:
# ── MODEL COMPLEXITY METRICS ──────────────────────────────────
print("\n" + "="*60)
print("MODEL COMPLEXITY METRICS")
print("="*60)

use_cuda = DEVICE.type == "cuda"

# ── FLOPs via thop ────────────────────────────────────────────
dummy_flops = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
macs, _     = thop_profile(model, inputs=(dummy_flops,), verbose=False)
flops_G     = round(macs * 2 / 1e9, 6)

# ── Disk size ──────────────────────────────────────────────────
with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
    tmp = f.name
torch.save(model.state_dict(), tmp)
size_mb = round(os.path.getsize(tmp) / 1024**2, 4)
os.remove(tmp)

# ── CPU inference timings ─────────────────────────────────────
print("\n⏱  Measuring CPU inference times ...")
model_cpu = BranchyViT(num_classes=NUM_CLASSES)
model_cpu.load_state_dict(torch.load(SAVE_PATH, map_location="cpu"))
model_cpu.eval()

dummy_single_cpu = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
dummy_batch_cpu  = torch.randn(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE)

with torch.no_grad():
    for _ in range(10):
        model_cpu(dummy_single_cpu)

times_cpu_single = []
with torch.no_grad():
    for _ in range(100):
        t0 = time.perf_counter()
        model_cpu(dummy_single_cpu)
        times_cpu_single.append(time.perf_counter() - t0)
inf_ms_single_cpu = np.mean(times_cpu_single) * 1000

with torch.no_grad():
    for _ in range(5):
        model_cpu(dummy_batch_cpu)

times_cpu_batch = []
with torch.no_grad():
    for _ in range(20):
        t0 = time.perf_counter()
        model_cpu(dummy_batch_cpu)
        times_cpu_batch.append(time.perf_counter() - t0)
inf_ms_batch128_cpu     = np.mean(times_cpu_batch) * 1000
inf_ms_per_img_cpu      = inf_ms_batch128_cpu / BATCH_SIZE
throughput_imgs_sec_cpu = BATCH_SIZE / (inf_ms_batch128_cpu / 1000)

del model_cpu, dummy_single_cpu, dummy_batch_cpu
print("✓ CPU timing done")

# ── GPU inference timings ─────────────────────────────────────
print("⏱  Measuring GPU inference times ...")
dummy_single_gpu = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
dummy_batch_gpu  = torch.randn(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

if use_cuda:
    with torch.no_grad():
        for _ in range(50):
            model(dummy_single_gpu)
    torch.cuda.synchronize()

    times_gpu_single = []
    with torch.no_grad():
        for _ in range(500):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(dummy_single_gpu)
            torch.cuda.synchronize()
            times_gpu_single.append(time.perf_counter() - t0)
    inf_ms_single_gpu = np.mean(times_gpu_single) * 1000

    start_ev = torch.cuda.Event(enable_timing=True)
    end_ev   = torch.cuda.Event(enable_timing=True)
    with torch.no_grad():
        for _ in range(10):
            model(dummy_batch_gpu)
    torch.cuda.synchronize()

    batch_times_gpu = []
    with torch.no_grad():
        for _ in range(100):
            start_ev.record()
            model(dummy_batch_gpu)
            end_ev.record()
            torch.cuda.synchronize()
            batch_times_gpu.append(start_ev.elapsed_time(end_ev))

    inf_ms_batch128_gpu     = np.mean(batch_times_gpu)
    inf_ms_per_img_gpu      = inf_ms_batch128_gpu / BATCH_SIZE
    throughput_imgs_sec_gpu = BATCH_SIZE / (inf_ms_batch128_gpu / 1000)
else:
    # CPU fallback (same as CPU block, run once)
    inf_ms_single_gpu       = inf_ms_single_cpu
    inf_ms_batch128_gpu     = inf_ms_batch128_cpu
    inf_ms_per_img_gpu      = inf_ms_per_img_cpu
    throughput_imgs_sec_gpu = throughput_imgs_sec_cpu
print("✓ GPU timing done")

# ── Adaptive inference timing at τ=0.8 ────────────────────────
dummy_single_gpu = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
with torch.no_grad():
    for _ in range(3):
        model.adaptive_forward(dummy_single_gpu, threshold=0.8)

if use_cuda:
    torch.cuda.synchronize()
    start_ev = torch.cuda.Event(enable_timing=True)
    end_ev   = torch.cuda.Event(enable_timing=True)
    start_ev.record()
    with torch.no_grad():
        for _ in range(50):
            model.adaptive_forward(dummy_single_gpu, threshold=0.8)
    end_ev.record()
    torch.cuda.synchronize()
    inf_ms_adapt = start_ev.elapsed_time(end_ev) / 50
else:
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(50):
            model.adaptive_forward(dummy_single_gpu, threshold=0.8)
    inf_ms_adapt = (time.perf_counter() - t0) / 50 * 1000

print(f"\n  Parameters               : {total_params:,}")
print(f"  Model size               : {size_mb:.4f} MB")
print(f"  FLOPs                    : {flops_G:.4f} G  (at {IMG_SIZE}×{IMG_SIZE})")
print(f"  --- Inference (CPU) ---")
print(f"  Single image CPU         : {inf_ms_single_cpu:.3f} ms")
print(f"  Batch-128 CPU            : {inf_ms_batch128_cpu:.2f} ms")
print(f"  Per-image CPU            : {inf_ms_per_img_cpu:.3f} ms")
print(f"  Throughput CPU           : {throughput_imgs_sec_cpu:.1f} imgs/sec")
print(f"  --- Inference (GPU) ---")
print(f"  Single image GPU         : {inf_ms_single_gpu:.3f} ms  (synchronized)")
print(f"  Batch-128 GPU            : {inf_ms_batch128_gpu:.2f} ms  (CUDA events)")
print(f"  Per-image GPU            : {inf_ms_per_img_gpu:.3f} ms")
print(f"  Throughput GPU           : {throughput_imgs_sec_gpu:.1f} imgs/sec")
print(f"  Inference (τ=0.8 adapt)  : {inf_ms_adapt:.3f} ms  (single, GPU)")

In [ ]:
# ── SAVE METRICS JSON (format mirrors __4__baseline_metrics_v3.json) ─
best_adaptive = max(adaptive_results, key=lambda r: r["accuracy"])

branchynet_vit_metrics = {
    "model"       : "branchynet_vit_small_patch4_32",
    "experiment"  : "branchynet_adaptive_computation",
    "method"      : "early_exit",
    "variant"     : "BranchyViT_Small",
    "original_arch": "vit_small_patch16_224",
    "dataset"     : "CIFAR-10",
    "train_device": str(DEVICE),
    "epochs"      : EPOCHS,
    "branch_weights"         : BRANCH_WEIGHTS,
    "exit_positions"         : [
        f"after block {model.exit1_block_idx} of 11 (~33% depth)",
        f"after block {model.exit2_block_idx} of 11 (~67% depth)",
        "final head (full depth)",
    ],
    "num_exits"              : 3,
    "exit_thresholds_tested" : EXIT_THRESHOLDS,
    # ── Standard (final-exit-only) metrics ────────────────────
    "accuracy"   : round(acc_final,  6),
    "precision"  : round(prec_final, 6),
    "recall"     : round(rec_final,  6),
    "f1"         : round(f1_final,   6),
    # ── Model complexity ───────────────────────────────────────
    "params"     : total_params,
    "size_mb"    : size_mb,
    "flops_G"    : flops_G,
    "input_resolution": IMG_SIZE,
    "patch_size" : 4,
    "num_patches": 64,
    # ── Inference timings (same structure as v3 baseline JSON) ─
    "inference_ms": {
        "cpu": {
            "single_img_ms"      : round(inf_ms_single_cpu, 4),
            "batch128_ms"        : round(inf_ms_batch128_cpu, 4),
            "per_img_ms"         : round(inf_ms_per_img_cpu, 4),
            "throughput_imgs_sec": round(throughput_imgs_sec_cpu, 1),
            "timing_method"      : "time.perf_counter()",
        },
        "gpu": {
            "single_img_ms"      : round(inf_ms_single_gpu, 4),
            "batch128_ms"        : round(inf_ms_batch128_gpu, 4),
            "per_img_ms"         : round(inf_ms_per_img_gpu, 4),
            "throughput_imgs_sec": round(throughput_imgs_sec_gpu, 1),
            "timing_method"      : (
                "CUDA events + torch.cuda.synchronize()"
                if use_cuda else "time.perf_counter() (CPU)"
            ),
        },
    },
    # ── Adaptive computation ───────────────────────────────────
    "inference_ms_adaptive_tau08"  : round(inf_ms_adapt, 4),
    "adaptive_threshold_results"   : adaptive_results,
    "best_adaptive_result"         : best_adaptive,
    # ── Housekeeping ───────────────────────────────────────────
    "saved_model_path": os.path.abspath(SAVE_PATH),
    "status"          : "ok",
}

output_json = "__5__branchynet_vit_metrics.json"
with open(output_json, "w") as f:
    json.dump(branchynet_vit_metrics, f, indent=2)
print(f"\n✓ Metrics saved to {output_json}")

In [ ]:
# ── COMPARISON SUMMARY ────────────────────────────────────────
print("\n" + "="*60)
print("COMPARISON SUMMARY: ViT Baseline vs BranchyViT")
print("="*60)

baseline_path = "__4__baseline_metrics_v3.json"
if os.path.exists(baseline_path):
    with open(baseline_path) as f:
        base = json.load(f)

    scalar_keys = ["accuracy", "precision", "recall", "f1", "params", "size_mb", "flops_G"]
    print(f"\n  {'Metric':<22} {'Baseline':>12} {'BranchyViT':>12} {'Δ':>10}")
    print("  " + "-"*58)
    for k in scalar_keys:
        bv = base.get(k, float('nan'))
        mv = branchynet_vit_metrics.get(k, float('nan'))
        if not isinstance(bv, (int, float)) or not isinstance(mv, (int, float)):
            continue
        delta = mv - bv
        sign  = "+" if delta > 0 else ""
        if k == "params":
            print(f"  {k:<22} {bv:>12,} {mv:>12,} {sign}{delta:>9,.0f}")
        else:
            print(f"  {k:<22} {bv:>12.4f} {mv:>12.4f} {sign}{delta:>9.4f}")

    # Inference comparison
    base_single_gpu = base.get("inference_ms", {}).get("gpu", {}).get("single_img_ms", None)
    if base_single_gpu:
        speedup = (1 - inf_ms_single_gpu / base_single_gpu) * 100
        print(f"\n  Inference single GPU — Baseline : {base_single_gpu:.3f} ms")
        print(f"  Inference single GPU — Branchy  : {inf_ms_single_gpu:.3f} ms  ({speedup:+.1f}%)")
    print(f"  Inference (τ=0.8 adapt)         : {inf_ms_adapt:.3f} ms  "
          f"(vs single full: {(1 - inf_ms_adapt/inf_ms_single_gpu)*100:+.1f}%)")
    print(f"\n  Best adaptive result (τ={best_adaptive['threshold']}):")
    print(f"    Accuracy   : {best_adaptive['accuracy']:.4f}")
    print(f"    Exit1      : {best_adaptive['frac_exit1']:.1%}")
    print(f"    Exit2      : {best_adaptive['frac_exit2']:.1%}")
    print(f"    Exit3      : {best_adaptive['frac_exit3']:.1%}")
    print(f"    Time/sample: {best_adaptive['avg_time_ms']:.4f} ms")
else:
    print(f"  (baseline JSON not found at {baseline_path} — skipping diff)")

In [ ]:
# ── SAVE FULL CHECKPOINT ──────────────────────────────────────
config_dict = {
    "model_name"    : "BranchyViT_Small_CIFAR10",
    "timm_base"     : "vit_small_patch16_224",
    "img_size"      : IMG_SIZE,
    "patch_size"    : 4,
    "num_patches"   : 64,
    "num_classes"   : NUM_CLASSES,
    "num_exits"     : 3,
    "exit_positions": [
        f"after block {model.exit1_block_idx}",
        f"after block {model.exit2_block_idx}",
        "final head",
    ],
    "branch_weights": BRANCH_WEIGHTS,
    "normalization" : {"mean": CIFAR_MEAN, "std": CIFAR_STD},
    "training": {
        "batch_size"      : BATCH_SIZE,
        "epochs"          : EPOCHS,
        "learning_rate"   : LR,
        "branch_lr"       : LR * 5,
        "optimizer"       : "AdamW",
        "weight_decay"    : 0.05,
        "scheduler"       : "WarmupCosine",
        "warmup_epochs"   : WARMUP_EPOCHS,
        "grad_clip"       : 1.0,
        "label_smoothing" : 0.1,
    }
}

torch.save({
    "model_state_dict": model.state_dict(),
    "config"          : config_dict,
    "classes"         : CIFAR10_CLASSES,
}, "__5__model_checkpoint.pth")

print("\n" + "="*60)
print("BRANCHYNET ViT + ADAPTIVE COMPUTATION — COMPLETE")
print("="*60)
print(f"  Weights    → {SAVE_PATH}")
print(f"  Checkpoint → __5__model_checkpoint.pth")
print(f"  Metrics    → {output_json}")